# Imported Libraries

In [273]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import f1_score


# Load audited data

In [274]:
news = pd.read_csv("../datasets/audited-data.csv")

news.head(10)

,title,type,content
0,Is life an ILLUSION? Researchers prove 'realit...,rumor,"Life is an illusion, at least on a quantum lev..."
1,REVEALED: This is where World War 3 will break...,rumor,"Dr Paul Miller, who served on the US’s Nationa..."
2,Qatar desperate to bolster MILITARY plan new R...,rumor,Fahad bin Mohammed Al-Attiyah is quoted as say...
3,Barcelona attack terrorists who ACCIDENTALLY b...,rumor,In the weeks before terrorists ploughed a van ...
4,Milan train crash: Horrific rush-hour derailme...,rumor,"The train, which was full of commuters, derail..."
5,Paedophile doctor Larry Nassar 'will never wal...,rumor,The pervert pleaded guilty to 10 counts of sex...
6,Turpin family: Parents in court shackled toget...,rumor,The children were tied up with ropes after one...
7,Milan train crash: Harrowing footage of derail...,rumor,The train was heading towards Milan before it ...
8,Mayon volcano eruption: SHOCK warning amid lav...,rumor,"The Mayon volcano, which stands at 8,077 foot ..."
9,EU NIGHTMARE: Italy's eurosceptics to unite sp...,rumor,European chiefs are seriously worried about th...


# Split Dataset into Training and Test Data
First pre-processing step is to split the audited dataset into training and test data to avoid data leakage; I've decided to split using the standard 80/20 split.

In [275]:
X = news[["title", "content"]]
y = news["type"]

X.head(10)

,title,content
0,Is life an ILLUSION? Researchers prove 'realit...,"Life is an illusion, at least on a quantum lev..."
1,REVEALED: This is where World War 3 will break...,"Dr Paul Miller, who served on the US’s Nationa..."
2,Qatar desperate to bolster MILITARY plan new R...,Fahad bin Mohammed Al-Attiyah is quoted as say...
3,Barcelona attack terrorists who ACCIDENTALLY b...,In the weeks before terrorists ploughed a van ...
4,Milan train crash: Horrific rush-hour derailme...,"The train, which was full of commuters, derail..."
5,Paedophile doctor Larry Nassar 'will never wal...,The pervert pleaded guilty to 10 counts of sex...
6,Turpin family: Parents in court shackled toget...,The children were tied up with ropes after one...
7,Milan train crash: Harrowing footage of derail...,The train was heading towards Milan before it ...
8,Mayon volcano eruption: SHOCK warning amid lav...,"The Mayon volcano, which stands at 8,077 foot ..."
9,EU NIGHTMARE: Italy's eurosceptics to unite sp...,European chiefs are seriously worried about th...


In [276]:
X.shape

(30000, 2)

In [277]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state = 4,
    test_size = 0.2)

In [278]:
X_train.shape

(24000, 2)

In [279]:
y_train.shape

(24000,)

# Minimal Preprocessing for Baseline Model
The baseline model I've chosen to use is `DummyClassifier`, with `strategy=most_frequent`; it will predict the value of `y_test` using the `type` value that most commonly appeared in `y_train`. Since `DummyClassifier` is just using the most frequent value that appears in `y_train` to predict the data, no preprocessing on the dataset is required for the simple baseline.

 First, I instantiated `DummyClassifier`.

In [280]:
model = DummyClassifier(strategy = 'most_frequent')


 Next, I set up stratified 10-fold cross-validation using `StratifiedKFold`, which will ensure that the proportion of article types is roughly equal across each fold.

In [281]:
cv = StratifiedKFold(
    n_splits = 10,
    shuffle=True,
    random_state=50
)

 Next I use `cross_validate()` to assess the baseline model on the training data. The model is fitted on nine folds and tested on the tenth. This process is repeated ten times so that the each fold is used for validation once. The mean of the ten Macro-F1 scores is 0.0130.

In [282]:
results = cross_validate(
    model,
    X_train,
    y_train,
    cv = cv,
    scoring = "f1_macro"
)

-I used the fitted model to generate predictions (`y_pred`) based on the values in `X_test`, then compared them to the actual values (`y_test`).


In [283]:
model.fit(X_train, y_train)

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
Name,Type,Value
"class_prior_ class_prior_: ndarray of shape (n_classes,) or list of such arraysFrequency of each class observed in `y`. For multioutput classificationproblems, this is computed independently for each output.","ndarray[float64](12,)","[0.08,0.08,0.08,...,0.08,0.08,0.08]"
"classes_ classes_: ndarray of shape (n_classes,) or list of such arraysUnique class labels observed in `y`. For multi-output classificationproblems, this attribute is a list of arrays as each output has anindependent set of possible classes.","ndarray[object](12,)","['bias','clickbait','conspiracy',...,'satire','unknown','unreliable']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](2,)","['title','content']"
n_classes_ n_classes_: int or list of intNumber of label for each output.,int,12
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,2
n_outputs_ n_outputs_: intNumber of outputs.,int,1
sparse_output_ sparse_output_: boolTrue if the array returned from predict is to be in sparse CSC format.Is automatically set to True if the input `y` is passed in sparseformat.,bool,False


In [284]:
y_pred = model.predict(X_test)

As you can see, the resulting Macro-F1 score from the baseline model is pretty low; 0.0122. This is because the `DummyClassifier` is only making predictions based on the most common article type in the dataset; `bias`. Since Macro-F1 applies equal weight to each of the 12 classes (`type`) in the dataset, the zero F1-scores for the other classes is lowering the score significantly.

In [285]:
test_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

print(f"Baseline Test Macro-F1: {test_f1:.4f}")

Baseline Test Macro-F1: 0.0122


In [286]:
print(pd.Series(y_pred).value_counts())

bias    6000
Name: count, dtype: int64
